# 4-Site-FMO OHNE Kompression auf echter IBM-Hardware

Dasselbe Ziel wie im Ordner `hardware_4_site_FMO`, aber **ohne Arnoldi**: der
Zustandsvektor ist der volle HEOM-Vektor aus System und allen ADOs, nicht seine
Projektion auf einen Krylov-Unterraum.

Damit verschwindet der Kompressionsfehler vollstaendig. Im komprimierten Lauf
blieben Site 3 und 4 bei null, obwohl qutip sie auf 0,13 bzw. 0,09 steigen
laesst — das war **nicht** die Hardware, sondern $\mathcal K_4$, das
$P^t y_0$ nur fuer $t \le 3$ exakt enthaelt. Ohne Kompression gibt es dieses
Problem nicht.

Der Preis ist die Registerbreite, und der ist brutal.

## 1a Warum das auf heutiger Hardware nicht laeuft

Der Propagator $P = e^{A\,\Delta t}$ ist eine **dichte** $n\times n$-Matrix auf
dem ganzen ADO-Raum. Die Zahl der ADOs waechst wie
$\binom{n_{\rm exp}+\text{Tiefe}}{\text{Tiefe}}$ mit
$n_{\rm exp} = n_{\rm sites}(N_k+1)$, und der Vektor hat $n = d^2 \cdot$ ADOs
Eintraege:

| Sites | Tiefe | ADOs | $n$ | gepolstert | Qubits |
|---|---|---|---|---|---|
| 2 | 1 | 5 | 20 | 32 | 6 |
| 2 | 2 | 15 | 60 | 64 | 7 |
| 4 | 1 | 9 | 144 | 256 | 9 |
| **4** | **2** | **45** | **720** | **1024** | **11** |

Eine allgemeine $q$-Qubit-Unitaere kostet $\sim 4^q/4$ Zweiqubit-Gatter.
Gemessen (transpiliert gegen die Heron-Basisgatter):

| Sites | Tiefe | Qubits | CZ **je Schritt** | Treue nach **einem** Schritt |
|---|---|---|---|---|
| 2 | 1 | 6 | 908 | 0,0624 |
| 2 | 2 | 7 | 3 692 | 0,0000 |
| 3 | 1 | 7 | 3 692 | 0,0000 |
| 4 | 2 | 11 | ~524 000 | 0,0000 |

Zum Vergleich: der komprimierte Lauf mit $m=4$ brauchte **138 CZ fuer alle
zehn Schritte** und erreichte Treue 0,656 bei RMS 0,043 gegen qutip.

**Der kleinste sinnvolle unkomprimierte Fall braucht fuer einen einzigen
Zeitschritt 6,6-mal mehr Zweiqubit-Gatter als der komprimierte Lauf fuer
zehn.** Es ist also kein einziger Zeitschritt erreichbar.

### 2 Sites statt 4 hilft hier sehr wohl

Mit Kompression brachte der Schritt von 4 auf 2 Sites **nichts**, weil dort
allein die Krylov-Dimension $m$ die Registerbreite bestimmt. Ohne Kompression
haengt sie direkt am ADO-Raum: 4 Sites brauchen 11 Qubits, 2 Sites nur 7 (bei
Tiefe 1 sogar 6). Das sind 4–5 Qubits weniger und damit ein Faktor
$4^4 = 256$ bis $4^5 = 1024$ an Gattern.

Reichen tut es trotzdem nicht — 908 CZ fuer einen Schritt bei einem Budget von
etwa 200.

### Tiefe 1 spart ein Qubit — und kostet mehr, als sie spart

Von Tiefe 2 auf Tiefe 1 faellt das Register von 7 auf 6 Qubits (2 Sites), also
von ~2048 auf ~908 CZ je Schritt. Aber Tiefe 1 ist physikalisch **nicht
konvergiert**. qutip `HEOMSolver`, 2 Sites, Abweichung von Tiefe 5 bei 200 fs:

| Tiefe | Qubits | Abweichung von Tiefe 5 |
|---|---|---|
| **1** | **6** | **0,12895** |
| 2 | 7 | 0,01155 |
| 3 | 8 | 0,02015 |

Das eine gesparte Qubit kostet 0,129 — **mehr als der gesamte
Kompressionsfehler von 0,038**, den der Verzicht auf Arnoldi beseitigen
sollte. Wer Qubits ueber die Abschneidetiefe spart, loest also ein anderes,
groeber genaehertes Problem.

Verglichen wird hier trotzdem konsequent gegen `HEOMSolver` mit **derselben**
Tiefe (`depth=grid['depth']` in `analyze`), damit der Algorithmusfehler sauber
vom Modellfehler getrennt bleibt. Die 0,129 sind Modellfehler, nicht
Verfahrensfehler — aber sie stehen trotzdem zwischen dir und der Wahrheit.

### Wie viele Zeitschritte tragen?

In Aer, 8192 Shots, gegen `HEOMSolver` gleicher Tiefe:

| t | T [fs] | $p_{\rm succ}$ | $N_{\rm acc}$ | Fehler (Tiefe 1) | Fehler (Tiefe 2) |
|---|---|---|---|---|---|
| 1 | 20 | 0,977 | 8002 | 0,00030 | 0,00081 |
| 5 | 100 | 0,708 | 5797 | 0,00013 | 0,00177 |
| 10 | 200 | 0,585 | 4792 | 0,01768 | 0,00568 |
| 15 | 300 | 0,498 | 4079 | 0,00060 | 0,00383 |
| 20 | 400 | 0,440 | 3600 | 0,00236 | 0,00170 |

**Der Fehler waechst nicht mit $t$.** Das ist der entscheidende Unterschied zur
Kompression, wo er von 0,031 bei $t=5$ auf 0,116 bei $t=25$ anstieg, weil
$\mathcal K_4$ nur die ersten drei Schritte exakt enthaelt.

Damit ist die Frage *„fuer wie viele Zeitschritte ist das so genau wie die
200 fs mit Kompression?"* beantwortet: **fuer alle.** Der komprimierte Lauf
hatte bei 200 fs 0,038 Algorithmusfehler (Aer) und 0,043 RMS auf Hardware;
unkomprimiert bleibt der Fehler ueber mindestens 20 Schritte unter 0,018 und
ist reines Shot-Rauschen. Die Nachselektion haelt bis $t=20$ noch 44 % der
Shots — auch sie ist nicht die Grenze.

Die Grenze ist allein die Gatterzahl, und die erlaubt **null** Schritte.

## 1b Was dieses Notebook trotzdem leistet

Die Schaltkreise sind **echt und vollstaendig hardwaretauglich**: nur
`unitary`, `measure`, `reset`, korrektes klassisches Register, Nachselektion,
aufgeschobene Messung, Twirling und XY4-Dynamical-Decoupling mit
ALAP-Scheduling. `run_on_backend` funktioniert — es weigert sich nur, wenn die
erwartete Gattertreue unter 1 % liegt, statt QPU-Zeit zu verbrennen
(`force=True` umgeht das).

In Aer laeuft alles korrekt durch, und dort sieht man den eigentlichen Punkt:
**ohne Kompression ist der Fehler reines Shot-Rauschen.**

In [ ]:
import os, sys, time
sys.path.insert(0, os.getcwd())
import numpy as np
import matplotlib.pyplot as plt
import hardware as hw

plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .3,
                     'axes.titlesize': 10, 'legend.fontsize': 8})
PIC = 'pictures'; os.makedirs(PIC, exist_ok=True)

# --- Parameter -----------------------------------------------------------
# N_SITES = 2 und DEPTH = 1 ist der EINZIGE Fall, der ueberhaupt in die Naehe
# von ausfuehrbar kommt (6 Qubits, 908 CZ je Schritt).  N_SITES = 4 mit
# DEPTH = 2 sind 11 Qubits und ~524000 CZ je Schritt -- in Aer richtig, auf
# Hardware chancenlos.
N_SITES = 2
DEPTH   = 1                        # HEOM-Abschneidetiefe
NK      = 1                        # Matsubara/Pade-Terme
DT_FS   = 20.0
TIMES   = list(range(1, 11))
METHOD  = 'svd'                    # 'svd' (empfohlen) oder 'sznagy'
DEFER   = True                     # aufgeschobene Messung
PAIRS   = [(0, 1)]                 # Kohaerenzen; je Paar +2 Kreise je Zeit
SHOTS   = 4096
BACKEND = 'ibm_kingston'

MODEL, RHO0 = hw.make_model(N_SITES, lam=35.0)
grid = hw.build_hardware_grid(n_sites=N_SITES, dt_fs=DT_FS, depth=DEPTH,
                              Nk=NK, model=MODEL, rho0=RHO0)
N    = grid['d']
cols = plt.cm.tab10(np.arange(N))

## 2 Die Schaltkreise

Ein Unterschied zum komprimierten Modul, diesmal zum Vorteil: ohne Kompression
sind die Ablesezeilen der Populationen **skalierte Einheitsvektoren**. Misst
man das Register in der Rechenbasis, ist das Histogramm bereits
$|x_k|^2$ fuer alle $k$ gleichzeitig — **ein** Schaltkreis liefert also alle
$d$ Populationen. Nur die Kohaerenzen brauchen noch je eine Basisdrehung.

Kreise je Zeitschritt: $1 + 2\,|\text{PAIRS}|$ statt $d + 2\,|\text{PAIRS}|$.

In [ ]:
jobs = hw.hardware_circuits(grid, TIMES, PAIRS, method=METHOD, defer=DEFER)
print(f"  {len(jobs)} Schaltkreise = {len(jobs)//len(TIMES)} je Zeit "
      f"({len(TIMES)} Zeiten)")
jobs[2]['circuit'].draw('mpl', fold=-1)

## 3 Was der Lauf kostet — **vor** dem Absenden

`report_cost` transpiliert wirklich, solange das Register hoechstens
`max_qubits_synth` Qubits hat. Darueber waere die Synthese einer allgemeinen
Unitaeren selbst exponentiell teuer (Minuten bis Stunden, viele GB), deshalb
wird dann die Schranke $2\cdot 4^{q-1}/4$ CZ je Schritt eingesetzt — eine
**Untergrenze**, die der Transpiler in der Praxis nicht erreicht.

`rep_delay`, `t_meas` und `t_reset` sind an einem echten Lauf kalibriert (Job
`da833pe0ukec7383ughg`: 42 s laut `job.usage()`, davon 3,7 s Gatterzeit).

In [ ]:
kosten = hw.report_cost(grid, jobs, shots=SHOTS)

## 4 Kostenlos gegenpruefen — und der eigentliche Punkt

In Aer gibt es kein Gatterrauschen. Weil hier **nicht komprimiert** wird, ist
der einzige verbleibende Fehler die Shot-Statistik plus die
HEOM-Abschneidung, die qutip genauso hat. Die Abweichung sollte also mit
$1/\sqrt{N_{\rm shots}}$ skalieren und sonst nichts.

Genau das unterscheidet dieses Modul vom komprimierten: dort waechst der
Fehler mit der Zahl der Zeitschritte, weil $\mathcal K_m$ nur die ersten $m$
Schritte exakt enthaelt. Hier waechst er **nicht**.

In [ ]:
rho_aer, err_aer, info_aer = hw.verify_in_aer(grid, jobs, shots=SHOTS,
                                             pairs=PAIRS)

## 5 Auf die echte Maschine

Der Token liegt in `~/.qiskit/qiskit-ibm.json` und muss **nicht** hier stehen.

`run_on_backend` bricht ab, wenn die erwartete Gattertreue des tiefsten
Kreises unter 1 % liegt. Das ist bei jeder hier sinnvollen Einstellung der
Fall — der Abbruch ist also das erwartete Verhalten und keine Fehlfunktion.
Mit `force=True` wird trotzdem abgesendet.

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

TOKEN = ""
if TOKEN:
    QiskitRuntimeService.save_account(token=TOKEN, overwrite=True)
    print("  neues Konto hinterlegt")
else:
    try:
        QiskitRuntimeService()
        print("  gespeichertes Konto gefunden -- kein Token noetig")
    except Exception as e:
        print(f"  KEIN Konto: {e}")

COUNTS_FILE = 'counts_without_compression.npy'

# ACHTUNG: verbraucht QPU-Zeit.
LOSSCHICKEN = False          # auf True setzen, wenn du bereit bist
FORCE       = False          # Treuepruefung uebergehen

if LOSSCHICKEN:
    if os.path.exists(COUNTS_FILE):
        sich = COUNTS_FILE.replace('.npy',
                                   f"_{time.strftime('%Y%m%d_%H%M%S')}.npy")
        os.rename(COUNTS_FILE, sich)
        print(f"  vorheriger Lauf gesichert als {sich}")
    counts = hw.run_on_backend(jobs, backend_name=BACKEND, shots=SHOTS,
                               force=FORCE)
    np.save(COUNTS_FILE, np.array(counts, dtype=object), allow_pickle=True)
    print(f"  Zaehlraten gesichert in {COUNTS_FILE}")
elif os.path.exists(COUNTS_FILE):
    counts = list(np.load(COUNTS_FILE, allow_pickle=True))
    if len(counts) != len(jobs):
        print(f"  ACHTUNG: {len(counts)} Zaehlraten, aber {len(jobs)} "
              f"Schaltkreise -- alter Lauf passt NICHT.  Wird ignoriert.")
        counts = None
    else:
        print(f"  {len(counts)} gespeicherte Zaehlraten geladen")
else:
    counts = None
    print("  Noch nichts abgeschickt.  LOSSCHICKEN = True setzen, wenn bereit.")

## 6 Auswerten und gegen qutip stellen

Verglichen wird ausschliesslich gegen qutips `HEOMSolver` mit
`DrudeLorentzPadeBath` (ein Bad je Site), damit man sieht, wie weit die
Loesung tatsaechlich von der exakten entfernt ist.

In [ ]:
if counts is not None:
    rho_hw, err_hw, info_hw = hw.analyze(grid, jobs, counts, pairs=PAIRS)
    hw_tr = info_hw['rho_tr']
else:
    rho_hw = err_hw = info_hw = hw_tr = None
    print("  !! KEINE Hardwaredaten -- es wird NUR Aer geplottet.")

ts   = info_aer['t_index']
ref  = info_aer['reference']
t_ex = np.arange(max(TIMES) + 1) * grid['dt_fs']
t_pt = ts * grid['dt_fs']
ok_a = np.isfinite(np.real(rho_aer[:, 0, 0]))
aer_tr = info_aer['rho_tr']

fig, ax = plt.subplots(1, 3, figsize=(15.6, 4))

for j in range(N):                          # (a) Populationen
    ax[0].plot(t_ex, np.real(ref[:, j, j]), color=cols[j], lw=1.7,
               label=f'Site {j+1}')
    ax[0].errorbar(t_pt[ok_a], np.real(aer_tr[ok_a, j, j]),
                   yerr=err_aer[ok_a, j, j], fmt='o', ms=5, capsize=3,
                   color=cols[j], mfc='white', lw=1.1)
    if rho_hw is not None:
        ax[0].errorbar(t_pt, np.real(hw_tr[:, j, j]), yerr=err_hw[:, j, j],
                       fmt='s', ms=5, capsize=3, color=cols[j], lw=1.1)
ax[0].set_ylabel('Population')
ax[0].set_title('Linie: qutip HEOMSolver.  Kreis: Aer.'
                + ('  Quadrat: QPU.' if rho_hw is not None else ''))
ax[0].legend(ncol=2, fontsize=7)

a, b = PAIRS[0]                             # (b) Kohaerenz
for teil, lab, mk, cc, ee in ((np.real, r'\mathrm{Re}', 'o', 'C0', err_aer),
                              (np.imag, r'\mathrm{Im}', 's', 'C2',
                               info_aer['rho_err_im'])):
    ax[1].plot(t_ex, teil(ref[:, a, b]), lw=1.7, color=cc,
               label=rf'${lab}\,\rho_{{{a+1}{b+1}}}$')
    ax[1].errorbar(t_pt[ok_a], teil(aer_tr[ok_a, a, b]), yerr=ee[ok_a, a, b],
                   fmt=mk, ms=5, capsize=3, mfc='white', lw=1.1, color=cc)
ax[1].axhline(0, color='0.85', lw=.8, zorder=0)
ax[1].set_title(rf'Kohaerenz $\rho_{{{a+1}{b+1}}}$ (++/RR-Drehung)')
ax[1].legend(fontsize=7)

dmax = lambda R, k: np.abs(np.real(np.diag(R[k]))
                           - np.real(np.diag(ref[ts[k]]))).max()
ax[2].semilogy(t_pt[ok_a], [dmax(aer_tr, k) for k in np.where(ok_a)[0]],
               'o-', color='C0', label='ohne Kompression (Aer)')
ax[2].axhline(1 / (2 * np.sqrt(SHOTS)), color='0.5', ls='--', lw=1,
              label=r'Shot-Rauschen $1/(2\sqrt{N})$')
ax[2].set_ylabel(r'max$_j\,|\rho_{jj} - $qutip$|$')
ax[2].set_title('Der Fehler waechst NICHT mit t')
ax[2].legend(fontsize=7)

for A in ax:
    A.set_xlabel('t [fs]'); A.grid(alpha=.25)
fig.suptitle(f'{N_SITES}-Site-FMO OHNE Kompression, {grid["n_qubits"]} Qubits, '
             f'Tiefe {DEPTH}, {SHOTS} Shots', fontsize=11)
fig.tight_layout()
fig.savefig(f'{PIC}/without_compression.png', dpi=150, bbox_inches='tight')
plt.show()

## 7 Fazit

Ohne Kompression ist der Algorithmus **exakt** — der verbleibende Fehler ist
Shot-Rauschen und die HEOM-Abschneidung, die qutip genauso hat. Er waechst
nicht mit der Zahl der Zeitschritte, und Site 3 und 4 kommen richtig heraus.

Bezahlt wird das mit der Registerbreite, und der Preis ist auf heutiger
Hardware nicht aufzubringen: schon ein einziger Zeitschritt kostet mehr
Zweiqubit-Gatter, als der komprimierte Zehn-Schritt-Lauf insgesamt gebraucht
hat.

Die Kompression ist damit nicht ein Zugestaendnis, sondern **die einzige
Moeglichkeit, das Verfahren ueberhaupt auf eine QPU zu bringen** — und ihr
Fehler (Site 3 und 4 bleiben liegen) ist der Preis dafuer, nicht ein Mangel
der Implementierung.